# NeuroMotion — First Model Exploration (Milestone 4 preparation)

**Task:** classify recorded EEG windows as **Rest**, **Left-hand motor imagery**, or **Right-hand motor imagery**. The classifier is an exploratory model for replaying public recordings in a virtual interaction; it is not a medical device and does not control a real prosthesis.

This notebook documents the initial six-step experiment and a controlled follow-up motivated by its observed failure mode. The initial run completed, but its validation predictions show that EEGNet predicted **Rest for every window**. Raw accuracy (0.50) only matches the training-majority baseline; balanced accuracy is 0.333 and macro-F1 is 0.222. The 0.60 and 0.70 confidence gates both rejected all 30 windows. These are genuine negative results, not a usable command model.

## Experiment design

- **AI role:** three-class EEG window classification.
- **Library / first candidate:** Braindecode `EEGClassifier` with the compact **EEGNet** architecture, trained from scratch using PyTorch.
- **Alternative candidates:** Braindecode `ShallowFBCSPNet` and a non-deep-learning CSP + LDA baseline remain future comparisons, not measured results.
- **Dataset:** PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB) v1.0.0.
- **Preprocessing:** EEG channels only; 4–38 Hz band-pass; 3-second windows from 0.5 to 3.5 seconds after `T0`, `T1`, or `T2`; convert V to microvolts; standardize each channel using mean and standard deviation fit **on training data only**.
- **Split:** subject S001, R04 for training, R08 for development/validation, and R12 reserved for a future untouched final test. The split is by complete run, not overlapping windows. R12 is not loaded in this notebook.
- **Initial configuration:** 10 epochs, batch size 8, Adam learning rate 0.001.
- **Gate diagnostic:** compare 0.60 and 0.70 using the same initial-model probabilities. Because no windows passed either gate, this comparison does **not** identify a useful threshold.

## Step 1 — define the role of AI and first candidate

The learned component maps a preprocessed EEG segment to one of three learned labels. A deterministic rule outside the model turns a low-confidence prediction into `NO COMMAND`. A confident `Rest` prediction is a valid class and means no movement; it is not the same as a rejected low-confidence result. The live replay layer must never execute a movement after `NO COMMAND`.

| Candidate | Why consider it | Decision for this experiment |
|---|---|---|
| Braindecode EEGNet | Compact EEG-specific temporal and spatial convolutional model; documented Braindecode training API | **Selected** for the first end-to-end neural model |
| Braindecode ShallowFBCSPNet | A second EEG model designed around temporal filtering and spatial features; useful for motor imagery | Future comparison after the pipeline works |
| CSP + LDA | Classical and relatively data-efficient motor-imagery baseline | Future benchmark; do not call it measured yet |

EEGNet is not pretrained here: its weights are initialized and fitted to the training data. The model is a starting point, not a claim that deep learning will outperform the classical baseline on this small within-subject split.

## Step 2 — environment and reproducibility

Use a dedicated Python 3.11 environment. This avoids mixing the project dependencies with the Python 3.14 environment where plotting/native-library issues were reported earlier. This notebook deliberately uses tables and printed diagnostics rather than Matplotlib plots.

In [1]:
import json
import random
import time
import platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import mne
import sklearn
import braindecode
from braindecode import EEGClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("MNE:", mne.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("Braindecode:", braindecode.__version__)
print("Device:", DEVICE)

Python: 3.12.15
MNE: 1.13.2
scikit-learn: 1.9.1
PyTorch: 2.14.1+cu130
Braindecode: 1.8.1
Device: cuda


c:\Users\cdsch\miniconda3\envs\bci\Lib\site-packages\braindecode\models\eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
c:\Users\cdsch\miniconda3\envs\bci\Lib\site-packages\braindecode\models\eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


## Step 2 — configure and verify the local dataset path

Keep raw data outside Git. Point `DATA_ROOT` to the folder containing subject folders such as `S001/S001R04.edf`. Common locations are searched as a convenience, but the notebook never downloads data silently.

In [2]:
SUBJECT = "S001"
TRAIN_RUNS = ("R04",)
VALIDATION_RUNS = ("R08",)
RESERVED_TEST_RUNS = ("R12",)  # Do not load or inspect this run during this homework.

# Change this to the parent folder containing S001, S002, ... if necessary.
DATA_ROOT = Path("data")
SEARCH_ROOTS = [
    DATA_ROOT,
    Path("eegmmidb"),
    Path("eegmmidb/1.0.0"),
    Path.home() / "mne_data" / "MNE-eegbci-data" / "files" / "eegmmidb" / "1.0.0",
    Path.home() / "Downloads" / "eegmmidb" / "1.0.0",
]

# Use the first candidate tree that contains the required training recording.
for root in SEARCH_ROOTS:
    if root.exists() and any(root.rglob(f"{SUBJECT}{TRAIN_RUNS[0]}.edf")):
        DATA_ROOT = root
        break


def find_recording(subject: str, run: str) -> Path:
    filename = f"{subject}{run}.edf"
    matches = sorted(DATA_ROOT.rglob(filename)) if DATA_ROOT.exists() else []
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Set DATA_ROOT to the folder containing "
            f"the subject folders. Current DATA_ROOT: {DATA_ROOT.resolve()}"
        )
    if len(matches) > 1:
        print(f"Warning: found {len(matches)} copies of {filename}; using {matches[0]}")
    return matches[0]

required_files = {
    run: find_recording(SUBJECT, run)
    for run in (*TRAIN_RUNS, *VALIDATION_RUNS)
}
print("Dataset root:", DATA_ROOT.resolve())
print("Subject:", SUBJECT)
for run, path in required_files.items():
    print(f"{run}: {path}")
print("Reserved final-test run(s) not loaded:", ", ".join(RESERVED_TEST_RUNS))

Dataset root: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\data
Subject: S001
R04: data\S001\S001R04.edf
R08: data\S001\S001R08.edf
Reserved final-test run(s) not loaded: R12


## Step 2 — load, preprocess, epoch, and check labels

PhysioNet's `T0`, `T1`, and `T2` meanings depend on run type. These are the unilateral **imagined** fist runs: `T0 = Rest`, `T1 = Left Hand imagery`, and `T2 = Right Hand imagery`. Each run is loaded and epoched independently. Any missing event type, non-finite samples, unexpected sampling rate, or channel mismatch raises an error rather than being silently repaired.

The epoch is 0.5–3.5 seconds after each event onset. Events that do not have enough following data may be dropped by MNE; the notebook reports requested versus retained windows.

In [3]:
EVENT_TO_LABEL = {"T0": 0, "T1": 1, "T2": 2}
CLASS_NAMES = ["Rest", "Left Hand", "Right Hand"]
CLASS_TO_LABEL = {name: idx for idx, name in enumerate(CLASS_NAMES)}

LOW_CUT_HZ = 4.0
HIGH_CUT_HZ = 38.0
WINDOW_TMIN = 0.5
WINDOW_TMAX = 3.5
EXPECTED_SFREQ = 160.0


def load_run(subject: str, run: str):
    path = find_recording(subject, run)
    raw = mne.io.read_raw_edf(path, preload=True, verbose="ERROR")
    sfreq = float(raw.info["sfreq"])
    if not np.isclose(sfreq, EXPECTED_SFREQ):
        raise ValueError(f"Unexpected sampling rate {sfreq} Hz in {path}; expected {EXPECTED_SFREQ} Hz.")

    raw.pick(picks="eeg")
    if len(raw.ch_names) == 0:
        raise ValueError(f"No EEG channels were found in {path}.")
    channel_names = list(raw.ch_names)
    raw.filter(l_freq=LOW_CUT_HZ, h_freq=HIGH_CUT_HZ, picks="eeg", verbose="ERROR")

    events, found_event_id = mne.events_from_annotations(
        raw, regexp=r"^(T0|T1|T2)$", verbose="ERROR"
    )
    missing = sorted(set(EVENT_TO_LABEL) - set(found_event_id))
    if missing:
        raise ValueError(f"Missing expected annotations {missing} in {path}; found {found_event_id}.")
    selected_event_id = {name: found_event_id[name] for name in EVENT_TO_LABEL}

    epochs = mne.Epochs(
        raw,
        events,
        event_id=selected_event_id,
        tmin=WINDOW_TMIN,
        tmax=WINDOW_TMAX,
        baseline=None,
        picks="eeg",
        preload=True,
        reject_by_annotation=True,
        verbose="ERROR",
    )
    X = epochs.get_data().astype(np.float32) * 1e6  # MNE stores EEG in volts; model uses microvolts.
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite EEG values found in {path}; inspect quality before training.")

    code_to_label = {selected_event_id[event_name]: EVENT_TO_LABEL[event_name]
                     for event_name in EVENT_TO_LABEL}
    code_to_event = {selected_event_id[event_name]: event_name
                     for event_name in EVENT_TO_LABEL}
    y = np.asarray([code_to_label[int(code)] for code in epochs.events[:, 2]], dtype=np.int64)
    onsets = (epochs.events[:, 0] - raw.first_samp) / sfreq
    metadata = pd.DataFrame({
        "subject": subject,
        "run": run,
        "event_onset_sec": np.round(onsets, 3),
        "annotation": [code_to_event[int(code)] for code in epochs.events[:, 2]],
        "label_id": y,
        "true_class": [CLASS_NAMES[int(value)] for value in y],
    })
    report = {
        "subject": subject,
        "run": run,
        "file": str(path),
        "sampling_rate_hz": sfreq,
        "channels": len(channel_names),
        "requested_events_T0_T1_T2": len(events),
        "retained_epochs": len(epochs),
        "dropped_epochs": len(events) - len(epochs),
        "samples_per_epoch": int(X.shape[2]) if len(X) else 0,
        "finite_values": bool(np.isfinite(X).all()),
        "min_uV": float(np.min(X)) if X.size else np.nan,
        "max_uV": float(np.max(X)) if X.size else np.nan,
    }
    return X, y, metadata, channel_names, sfreq, report

run_data = {}
run_reports = []
for run in (*TRAIN_RUNS, *VALIDATION_RUNS):
    loaded = load_run(SUBJECT, run)
    run_data[run] = loaded
    run_reports.append(loaded[-1])

quality_summary = pd.DataFrame(run_reports)
display(quality_summary)

reference_channels = run_data[TRAIN_RUNS[0]][3]
for run in (*TRAIN_RUNS, *VALIDATION_RUNS):
    if run_data[run][3] != reference_channels:
        raise ValueError(f"Channel names/order differ in {run}. Harmonize channels explicitly before training.")
    if not np.isclose(run_data[run][4], EXPECTED_SFREQ):
        raise ValueError(f"Unexpected sampling rate in {run}.")

X_train_raw = np.concatenate([run_data[run][0] for run in TRAIN_RUNS], axis=0)
y_train = np.concatenate([run_data[run][1] for run in TRAIN_RUNS], axis=0)
meta_train = pd.concat([run_data[run][2] for run in TRAIN_RUNS], ignore_index=True)
X_val_raw = np.concatenate([run_data[run][0] for run in VALIDATION_RUNS], axis=0)
y_val = np.concatenate([run_data[run][1] for run in VALIDATION_RUNS], axis=0)
meta_val = pd.concat([run_data[run][2] for run in VALIDATION_RUNS], ignore_index=True)

if len(X_train_raw) == 0 or len(X_val_raw) == 0:
    raise ValueError("No training or validation windows were retained. Check annotations/window timing.")
if set(np.unique(y_train)) != set(range(len(CLASS_NAMES))):
    raise ValueError(f"Training split does not contain all three classes: {Counter(y_train)}")
if not set(np.unique(y_val)).issubset(set(range(len(CLASS_NAMES)))):
    raise ValueError("Unexpected validation labels.")

print("Train tensor shape (windows, channels, time):", X_train_raw.shape)
print("Validation tensor shape (windows, channels, time):", X_val_raw.shape)
print("Training class counts:")
display(pd.Series([CLASS_NAMES[int(v)] for v in y_train]).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("count").to_frame())
print("Validation class counts:")
display(pd.Series([CLASS_NAMES[int(v)] for v in y_val]).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("count").to_frame())

,subject,run,file,sampling_rate_hz,channels,requested_events_T0_T1_T2,retained_epochs,dropped_epochs,samples_per_epoch,finite_values,min_uV,max_uV
0,S001,R04,data\S001\S001R04.edf,160.0,64,30,30,0,481,True,-315.934937,305.064911
1,S001,R08,data\S001\S001R08.edf,160.0,64,30,30,0,481,True,-251.535553,296.216949


Train tensor shape (windows, channels, time): (30, 64, 481)
Validation tensor shape (windows, channels, time): (30, 64, 481)
Training class counts:


,count
Rest,15
Left Hand,8
Right Hand,7


Validation class counts:


,count
Rest,15
Left Hand,8
Right Hand,7


## Step 2 — fit normalization only on training data

This is the only learned preprocessing step in this notebook. Per-channel means and standard deviations are calculated from the training run only, then reused unchanged on validation. This helps keep the validation data from influencing model preparation.

In [4]:
channel_mean = X_train_raw.mean(axis=(0, 2), keepdims=True)
channel_std = X_train_raw.std(axis=(0, 2), keepdims=True)
channel_std = np.where(channel_std < 1e-6, 1.0, channel_std)

X_train = ((X_train_raw - channel_mean) / channel_std).astype(np.float32)
X_val = ((X_val_raw - channel_mean) / channel_std).astype(np.float32)

assert X_train.shape[1:] == X_val.shape[1:]
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

print("Normalization fitted on training windows only.")
print("Training normalized mean/std:", round(float(X_train.mean()), 4), round(float(X_train.std()), 4))
print("Validation normalized mean/std (not forced to 0/1):", round(float(X_val.mean()), 4), round(float(X_val.std()), 4))

Normalization fitted on training windows only.
Training normalized mean/std: 0.0 1.0
Validation normalized mean/std (not forced to 0/1): -0.0028 0.9418


## Step 3 — fit the first candidate end-to-end

The run uses a fixed seed, EEGNet, Adam, learning rate 0.001, batch size 8, and 10 epochs. Braindecode's `EEGClassifier` is given `train_split=None` because the run-level validation set is already separate; an internal random split of the training run is unnecessary for this first exploration.

In [5]:
INITIAL_CONFIG = {
    "model": "Braindecode EEGNet",
    "max_epochs": 10,
    "batch_size": 8,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "random_seed": SEED,
    "confidence_threshold_initial": 0.60,
    "confidence_threshold_revised": 0.70,
}

net = EEGClassifier(
    "EEGNet",
    module__n_chans=int(X_train.shape[1]),
    module__n_outputs=len(CLASS_NAMES),
    module__n_times=int(X_train.shape[2]),
    module__sfreq=float(EXPECTED_SFREQ),
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    lr=INITIAL_CONFIG["learning_rate"],
    max_epochs=INITIAL_CONFIG["max_epochs"],
    batch_size=INITIAL_CONFIG["batch_size"],
    train_split=None,
    iterator_train__shuffle=True,
    iterator_train__drop_last=False,
    device=DEVICE,
    verbose=1,
)

training_start = time.perf_counter()
net.fit(X_train, y_train)
training_seconds = time.perf_counter() - training_start
print(f"Completed initial training in {training_seconds:.2f} seconds on {DEVICE}.")

# Save the epoch history for the report. These are training logs, not validation scores.
OUTPUT_DIR = Path("outputs")
ARTIFACT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
history_df = pd.DataFrame(net.history.to_list())
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
display(history_df.tail(5))

  epoch    train_loss      dur
-------  ------------  -------
      1        1.0754  12.9500
      2        0.9996  0.0200
      3        0.9642  0.0240
      4        0.9554  0.0280
      5        0.9372  0.0220
      6        0.9088  0.0210
      7        0.8817  0.0210
      8        0.8762  0.0200
      9        0.8728  0.0210
     10        0.8263  0.0160
Completed initial training in 14.54 seconds on cuda.


,batches,epoch,train_batch_count,dur,train_loss,train_loss_best
5,"[{'train_loss': 0.979208767414093, 'train_batc...",6,4,0.020998,0.908812,True
6,"[{'train_loss': 0.9331609010696411, 'train_bat...",7,4,0.021000,0.881678,True
7,"[{'train_loss': 0.9035722017288208, 'train_bat...",8,4,0.020000,0.876245,True
8,"[{'train_loss': 0.8011072278022766, 'train_bat...",9,4,0.020999,0.872753,True
9,"[{'train_loss': 1.0406920909881592, 'train_bat...",10,4,0.015999,0.826321,True


## Step 4 — evaluate the initial model on the validation run

Report raw classification metrics before confidence gating, including class-balanced metrics because accuracy alone can hide a weak class. The simple majority-class baseline is calculated from training labels only and predicts that one class for all validation windows. These metrics are preliminary because this first split covers one subject and one training run.

In [6]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

probabilities = np.asarray(net.predict_proba(X_val), dtype=float)
if probabilities.shape != (len(y_val), len(CLASS_NAMES)):
    raise ValueError(f"Unexpected probability matrix shape: {probabilities.shape}")
if not np.isfinite(probabilities).all():
    raise ValueError("Model produced non-finite probabilities.")
if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-3):
    raise ValueError("predict_proba rows do not sum to approximately 1; inspect Braindecode/skorch configuration.")

y_pred = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)

model_raw_metrics = {
    "model": "EEGNet (raw prediction)",
    "validation_windows": int(len(y_val)),
    "accuracy": float(accuracy_score(y_val, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_val, y_pred)),
    "macro_f1": float(f1_score(y_val, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
}
majority_class = int(np.bincount(y_train, minlength=len(CLASS_NAMES)).argmax())
majority_pred = np.full_like(y_val, majority_class)
majority_metrics = {
    "model": f"Training majority baseline ({CLASS_NAMES[majority_class]})",
    "validation_windows": int(len(y_val)),
    "accuracy": float(accuracy_score(y_val, majority_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_val, majority_pred)),
    "macro_f1": float(f1_score(y_val, majority_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
}
raw_metric_table = pd.DataFrame([majority_metrics, model_raw_metrics])
display(raw_metric_table.round(4))

print("Classification report (validation only):")
print(classification_report(
    y_val, y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    zero_division=0,
))

confusion_df = pd.DataFrame(
    confusion_matrix(y_val, y_pred, labels=[0, 1, 2]),
    index=[f"true_{name}" for name in CLASS_NAMES],
    columns=[f"pred_{name}" for name in CLASS_NAMES],
)
print("Confusion matrix (counts):")
display(confusion_df)
raw_metric_table.to_csv(OUTPUT_DIR / "raw_validation_metrics.csv", index=False)
confusion_df.to_csv(OUTPUT_DIR / "raw_confusion_matrix.csv")

,model,validation_windows,accuracy,balanced_accuracy,macro_f1
0,Training majority baseline (Rest),30,0.5,0.3333,0.2222
1,EEGNet (raw prediction),30,0.5,0.3333,0.2222


Classification report (validation only):
              precision    recall  f1-score   support

        Rest       0.50      1.00      0.67        15
   Left Hand       0.00      0.00      0.00         8
  Right Hand       0.00      0.00      0.00         7

    accuracy                           0.50        30
   macro avg       0.17      0.33      0.22        30
weighted avg       0.25      0.50      0.33        30

Confusion matrix (counts):


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,15,0,0
true_Left Hand,8,0,0
true_Right Hand,7,0,0


## Step 5 — inspect weak outputs and quantify confidence-gate behavior

`Rest` remains a valid label and produces no movement. If the largest predicted class probability is below a threshold, the runtime emits `NO COMMAND` and must stop. A confidently wrong classification is not knowable from confidence alone at runtime; the validation labels let us count it offline.

The two threshold rows below use the **same trained model, same validation windows, and same probabilities**. Only the confidence threshold changes from 0.60 to 0.70.

In [7]:
def gate_metrics(threshold: float) -> dict:
    accepted = confidence >= threshold
    emitted_movement = accepted & np.isin(y_pred, [1, 2])
    true_rest = y_val == 0
    false_movement_on_rest = emitted_movement & true_rest
    wrong_movement = emitted_movement & (y_pred != y_val)

    return {
        "configuration": f"EEGNet + confidence gate {threshold:.2f}",
        "confidence_threshold": threshold,
        "validation_windows": int(len(y_val)),
        "raw_accuracy": float(accuracy_score(y_val, y_pred)),
        "raw_macro_f1": float(f1_score(y_val, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "accepted_output_rate": float(accepted.mean()),
        "no_command_rate": float((~accepted).mean()),
        "movement_command_rate": float(emitted_movement.mean()),
        "false_movement_on_rest_rate": (
            float(false_movement_on_rest.sum() / true_rest.sum()) if true_rest.any() else np.nan
        ),
        "wrong_movement_command_rate": (
            float(wrong_movement.sum() / emitted_movement.sum()) if emitted_movement.any() else np.nan
        ),
        "movement_commands": int(emitted_movement.sum()),
        "wrong_movement_commands": int(wrong_movement.sum()),
    }

experiment_results = pd.DataFrame([
    gate_metrics(INITIAL_CONFIG["confidence_threshold_initial"]),
    gate_metrics(INITIAL_CONFIG["confidence_threshold_revised"]),
])
display(experiment_results.round(4))
experiment_results.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)

# Make individual validation outcomes auditable; include every wrong prediction and low-confidence case.
inspection = meta_val.copy().reset_index(drop=True)
inspection["true_class"] = [CLASS_NAMES[int(v)] for v in y_val]
inspection["predicted_class"] = [CLASS_NAMES[int(v)] for v in y_pred]
inspection["confidence"] = confidence
inspection["correct_raw_prediction"] = (y_val == y_pred)
inspection["threshold_060_output"] = [
    "NO COMMAND" if p < INITIAL_CONFIG["confidence_threshold_initial"]
    else ("REST (no movement)" if cls == 0 else CLASS_NAMES[int(cls)])
    for p, cls in zip(confidence, y_pred)
]
inspection["threshold_070_output"] = [
    "NO COMMAND" if p < INITIAL_CONFIG["confidence_threshold_revised"]
    else ("REST (no movement)" if cls == 0 else CLASS_NAMES[int(cls)])
    for p, cls in zip(confidence, y_pred)
]
inspection["review_status_at_060"] = [
    "Low confidence - rejected" if p < INITIAL_CONFIG["confidence_threshold_initial"]
    else ("Confident misclassification" if true != pred else "Correct accepted output")
    for p, true, pred in zip(confidence, y_val, y_pred)
]
inspection.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)

needs_review = inspection[inspection["review_status_at_060"] != "Correct accepted output"]
if needs_review.empty:
    print("No incorrect or rejected windows at threshold 0.60; displaying the 10 least-confident examples instead.")
    review_rows = inspection.nsmallest(min(10, len(inspection)), "confidence")
else:
    review_rows = needs_review.sort_values(
        ["correct_raw_prediction", "confidence"], ascending=[True, True]
    ).head(10)
print(f"Unique validation windows inspected: {len(inspection)}; rows shown: {len(review_rows)}")
display(review_rows[[
    "run", "event_onset_sec", "true_class", "predicted_class", "confidence",
    "threshold_060_output", "threshold_070_output", "review_status_at_060"
]])

print("Wrong raw classifications:", int((y_val != y_pred).sum()), "of", int(len(y_val)))
print("Low-confidence rejections at 0.60:", int((confidence < 0.60).sum()), "of", int(len(y_val)))

,configuration,confidence_threshold,validation_windows,raw_accuracy,raw_macro_f1,accepted_output_rate,no_command_rate,movement_command_rate,false_movement_on_rest_rate,wrong_movement_command_rate,movement_commands,wrong_movement_commands
0,EEGNet + confidence gate 0.60,0.6,30,0.5,0.2222,0.0,1.0,0.0,0.0,NaN,0,0
1,EEGNet + confidence gate 0.70,0.7,30,0.5,0.2222,0.0,1.0,0.0,0.0,NaN,0,0


Unique validation windows inspected: 30; rows shown: 10


,run,event_onset_sec,true_class,predicted_class,confidence,threshold_060_output,threshold_070_output,review_status_at_060
1,R08,4.2,Left Hand,Rest,0.341435,NO COMMAND,NO COMMAND,Low confidence - rejected
25,R08,103.8,Right Hand,Rest,0.344778,NO COMMAND,NO COMMAND,Low confidence - rejected
19,R08,78.9,Left Hand,Rest,0.349486,NO COMMAND,NO COMMAND,Low confidence - rejected
9,R08,37.4,Left Hand,Rest,0.351597,NO COMMAND,NO COMMAND,Low confidence - rejected
3,R08,12.5,Right Hand,Rest,0.353253,NO COMMAND,NO COMMAND,Low confidence - rejected
21,R08,87.2,Left Hand,Rest,0.353916,NO COMMAND,NO COMMAND,Low confidence - rejected
29,R08,120.4,Left Hand,Rest,0.353922,NO COMMAND,NO COMMAND,Low confidence - rejected
11,R08,45.7,Right Hand,Rest,0.354531,NO COMMAND,NO COMMAND,Low confidence - rejected
27,R08,112.1,Left Hand,Rest,0.357949,NO COMMAND,NO COMMAND,Low confidence - rejected
5,R08,20.8,Left Hand,Rest,0.361016,NO COMMAND,NO COMMAND,Low confidence - rejected


Wrong raw classifications: 15 of 30
Low-confidence rejections at 0.60: 30 of 30


## Step 5 — measure one practical constraint: inference latency

This records CPU/GPU, number of unique validation windows, and single-window inference latency. To get a minimally useful timing distribution when the validation run is small, the timing probe repeats calls over existing validation windows until 100 calls have been timed. **Repeated calls are only for timing; accuracy and other model metrics above count each unique validation window once.** This is a preliminary local benchmark, not a claim of real-time system latency under every deployment condition.

In [8]:
# Warm up the model first so one-time initialization is not mixed into measured calls.
latency_calls = max(100, len(X_val))
for i in range(min(10, len(X_val))):
    _ = net.predict_proba(X_val[[i]])
if DEVICE == "cuda":
    torch.cuda.synchronize()

latency_ms = []
for i in range(latency_calls):
    x_one = X_val[[i % len(X_val)]]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = net.predict_proba(x_one)
    if DEVICE == "cuda":
        # CUDA operations are asynchronous; synchronize to measure completed inference.
        torch.cuda.synchronize()
    latency_ms.append((time.perf_counter() - t0) * 1000.0)

latency_summary = {
    "device": DEVICE,
    "unique_validation_windows": int(len(X_val)),
    "warmup_calls": int(min(10, len(X_val))),
    "timed_single_window_calls": int(latency_calls),
    "median_single_window_inference_ms": float(np.median(latency_ms)),
    "p95_single_window_inference_ms": float(np.percentile(latency_ms, 95)),
    "measurement_note": "Warm-up excluded; CUDA synchronized around each measured call; repeats validation windows only for timing if fewer than 100 unique windows exist.",
}
display(pd.DataFrame([latency_summary]).round(3))
with open(OUTPUT_DIR / "latency_summary.json", "w", encoding="utf-8") as f:
    json.dump(latency_summary, f, indent=2)


,device,unique_validation_windows,warmup_calls,timed_single_window_calls,median_single_window_inference_ms,p95_single_window_inference_ms,measurement_note
0,cuda,30,10,100,0.845,1.808,Warm-up excluded; CUDA synchronized around eac...


## Step 6 — interpret the confidence-gate diagnostic and save the initial handoff

The 0.60-versus-0.70 threshold comparison is an operational diagnostic, not a successful model improvement. In the saved initial run, all 30 validation windows fell below 0.60, so both thresholds produced `NO COMMAND`. This safely prevents movement, but it does not show that 0.70 is better; the model has not learned usable class separation yet.

The initial model handoff saves weights plus model dimensions, ordered channel names, filter/window parameters, class map, run IDs, and training-only normalization values. A future runtime must use the same ordered channels and exactly the same preprocessing/normalization. Do not deploy the initial weights as a usable controller while its validation output is collapsed to Rest/NO COMMAND.


In [9]:
weights_path = ARTIFACT_DIR / "neuromotion_eegnet_state_dict.pt"
config_path = ARTIFACT_DIR / "neuromotion_eegnet_config.json"
state_dict_cpu = {name: tensor.detach().cpu() for name, tensor in net.module_.state_dict().items()}
torch.save(state_dict_cpu, weights_path)

model_handoff = {
    "project": "NeuroMotion",
    "model_class": "braindecode.models.EEGNet",
    "classifier_wrapper": "braindecode.EEGClassifier",
    "braindecode_version": braindecode.__version__,
    "pytorch_version": torch.__version__,
    "subject": SUBJECT,
    "training_runs": list(TRAIN_RUNS),
    "validation_runs": list(VALIDATION_RUNS),
    "reserved_test_runs_not_used": list(RESERVED_TEST_RUNS),
    "class_mapping": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "channel_names_in_order": reference_channels,
    "sampling_rate_hz": EXPECTED_SFREQ,
    "bandpass_hz": [LOW_CUT_HZ, HIGH_CUT_HZ],
    "epoch_seconds_after_event": [WINDOW_TMIN, WINDOW_TMAX],
    "n_channels": int(X_train.shape[1]),
    "n_times": int(X_train.shape[2]),
    "input_units_before_standardization": "microvolts",
    "normalization": {
        "method": "per-channel z-score",
        "fit_split": "training run(s) only",
        "channel_mean_microvolts": channel_mean.reshape(-1).astype(float).tolist(),
        "channel_std_microvolts": channel_std.reshape(-1).astype(float).tolist(),
    },
    "training_configuration": INITIAL_CONFIG,
    "active_threshold_candidate": INITIAL_CONFIG["confidence_threshold_initial"],
    "revised_threshold_candidate_for_validation_comparison": INITIAL_CONFIG["confidence_threshold_revised"],
    "validation_raw_accuracy": model_raw_metrics["accuracy"],
    "validation_balanced_accuracy": model_raw_metrics["balanced_accuracy"],
    "validation_macro_f1": model_raw_metrics["macro_f1"],
    "notes": [
        "Exploratory within-subject model; not a medical device.",
        "Threshold selection remains a development decision, not a safety guarantee.",
        "Do not use the reserved R12 test run until model and threshold choices are frozen.",
        "Low confidence must lead to NO COMMAND and STOP; never execute the rejected prediction.",
    ],
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_handoff, f, indent=2)

print("Saved weights:", weights_path.resolve(), f"({weights_path.stat().st_size:,} bytes)")
print("Saved model config:", config_path.resolve())
print("Saved CSV/JSON outputs in:", OUTPUT_DIR.resolve())

Saved weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_state_dict.pt (19,318 bytes)
Saved model config: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_config.json
Saved CSV/JSON outputs in: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs


## Initial-experiment summary

The printed values and table below summarize the initial EEGNet run and its gate diagnostic. Interpret the result as a failed first baseline: accuracy matches the majority-class baseline, balanced accuracy equals the three-class chance reference (approximately 0.333), macro-F1 is low, and all validation outputs were rejected by both thresholds. The follow-up below tests class-weighted loss as a single targeted training change and compares both models on the same held-out validation run.


In [10]:
initial = experiment_results.iloc[0]
revised = experiment_results.iloc[1]
print(f"Initial model: EEGNet, {INITIAL_CONFIG['max_epochs']} epochs, Adam lr={INITIAL_CONFIG['learning_rate']}, batch={INITIAL_CONFIG['batch_size']}.")
print(f"Validation windows: {len(y_val)} from run(s) {', '.join(VALIDATION_RUNS)}; R12 remains untouched.")
print(f"Raw accuracy={model_raw_metrics['accuracy']:.3f}; balanced accuracy={model_raw_metrics['balanced_accuracy']:.3f}; macro-F1={model_raw_metrics['macro_f1']:.3f}.")
print(f"Gate 0.60: accepted={initial['accepted_output_rate']:.1%}; false movement on true-rest windows={initial['false_movement_on_rest_rate']:.1%}; wrong movement among movement commands={initial['wrong_movement_command_rate']:.1%}.")
print(f"Gate 0.70: accepted={revised['accepted_output_rate']:.1%}; false movement on true-rest windows={revised['false_movement_on_rest_rate']:.1%}; wrong movement among movement commands={revised['wrong_movement_command_rate']:.1%}.")
print("Interpretation prompt: Did 0.70 reduce false movements enough to justify its change in accepted-output rate? Mention the small single-subject/run-level sample and any confident errors listed above.")

Initial model: EEGNet, 10 epochs, Adam lr=0.001, batch=8.
Validation windows: 30 from run(s) R08; R12 remains untouched.
Raw accuracy=0.500; balanced accuracy=0.333; macro-F1=0.222.
Gate 0.60: accepted=0.0%; false movement on true-rest windows=0.0%; wrong movement among movement commands=nan%.
Gate 0.70: accepted=0.0%; false movement on true-rest windows=0.0%; wrong movement among movement commands=nan%.
Interpretation prompt: Did 0.70 reduce false movements enough to justify its change in accepted-output rate? Mention the small single-subject/run-level sample and any confident errors listed above.


## Controlled follow-up — test class-weighted loss

The saved baseline predicts `Rest` for all validation windows. Raising the confidence threshold cannot fix that failure; it only withholds more outputs. This follow-up tests **one training change only**: give each class a loss weight inversely proportional to its frequency in the training run. Both comparison models are re-created with the same seed and trained with the same EEGNet architecture, 10 epochs, optimizer, batch size, preprocessing, and whole-run split. The sole intended difference is the loss weights.

In [11]:
# Construct a paired comparison so the only intended change is class-weighted loss.
class_counts = np.bincount(y_train, minlength=len(CLASS_NAMES)).astype(np.float64)
if np.any(class_counts == 0):
    raise ValueError(f"Cannot compute class weights; a training class is missing: {class_counts}")
class_weight_values = len(y_train) / (len(CLASS_NAMES) * class_counts)
class_weight_tensor = torch.as_tensor(class_weight_values, dtype=torch.float32)

print("Training counts by class:", {CLASS_NAMES[i]: int(class_counts[i]) for i in range(len(CLASS_NAMES))})
print("Class weights:", {CLASS_NAMES[i]: round(float(class_weight_values[i]), 4) for i in range(len(CLASS_NAMES))})


def fit_comparison_eegnet(loss_weights=None):
    """Fit EEGNet with the same seed/settings; optionally weight CE loss by class."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    extra_criterion_kwargs = {}
    if loss_weights is not None:
        # Skorch passes criterion__weight to torch.nn.CrossEntropyLoss.
        extra_criterion_kwargs["criterion__weight"] = torch.as_tensor(
            loss_weights, dtype=torch.float32
        )

    model = EEGClassifier(
        "EEGNet",
        module__n_chans=int(X_train.shape[1]),
        module__n_outputs=len(CLASS_NAMES),
        module__n_times=int(X_train.shape[2]),
        module__sfreq=float(EXPECTED_SFREQ),
        criterion=torch.nn.CrossEntropyLoss,
        optimizer=torch.optim.Adam,
        lr=INITIAL_CONFIG["learning_rate"],
        max_epochs=INITIAL_CONFIG["max_epochs"],
        batch_size=INITIAL_CONFIG["batch_size"],
        train_split=None,
        iterator_train__shuffle=True,
        iterator_train__drop_last=False,
        device=DEVICE,
        verbose=0,
        **extra_criterion_kwargs,
    )
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    return model, elapsed


paired_unweighted, paired_unweighted_seconds = fit_comparison_eegnet(loss_weights=None)
paired_weighted, paired_weighted_seconds = fit_comparison_eegnet(loss_weights=class_weight_values)


def evaluate_model(model, model_name):
    probs = np.asarray(model.predict_proba(X_val), dtype=np.float64)
    if probs.shape != (len(y_val), len(CLASS_NAMES)) or not np.isfinite(probs).all():
        raise ValueError(f"Invalid validation probabilities from {model_name}: shape={probs.shape}")
    if not np.allclose(probs.sum(axis=1), 1.0, atol=1e-3):
        raise ValueError(f"Probability rows from {model_name} do not sum to 1.")
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    metrics = {
        "model": model_name,
        "validation_windows": int(len(y_val)),
        "accuracy": float(accuracy_score(y_val, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, pred)),
        "macro_f1": float(f1_score(y_val, pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "max_confidence": float(conf.max()),
        "accepted_rate_at_0.60": float((conf >= 0.60).mean()),
        "accepted_rate_at_0.70": float((conf >= 0.70).mean()),
    }
    return probs, pred, conf, metrics


paired_results = {}
paired_rows = []
for name, model, elapsed in [
    ("EEGNet unweighted (paired baseline)", paired_unweighted, paired_unweighted_seconds),
    ("EEGNet class-weighted CE", paired_weighted, paired_weighted_seconds),
]:
    probs, pred, conf, metrics = evaluate_model(model, name)
    metrics["training_seconds"] = float(elapsed)
    metrics["predicted_Rest"] = int((pred == 0).sum())
    metrics["predicted_Left_Hand"] = int((pred == 1).sum())
    metrics["predicted_Right_Hand"] = int((pred == 2).sum())
    paired_results[name] = {"model": model, "probabilities": probs, "prediction": pred, "confidence": conf}
    paired_rows.append(metrics)

paired_comparison = pd.DataFrame(paired_rows)
print("Paired comparison on the same R08 validation windows (not final-test results):")
display(paired_comparison.round(4))
paired_comparison.to_csv(OUTPUT_DIR / "class_weighted_model_comparison.csv", index=False)

for name, content in paired_results.items():
    print("\nConfusion matrix —", name)
    display(pd.DataFrame(
        confusion_matrix(y_val, content["prediction"], labels=[0, 1, 2]),
        index=[f"true_{label}" for label in CLASS_NAMES],
        columns=[f"pred_{label}" for label in CLASS_NAMES],
    ))
    print(classification_report(
        y_val, content["prediction"], labels=[0, 1, 2],
        target_names=CLASS_NAMES, zero_division=0,
    ))

# Evaluate the safety gate for both trained models. NaN wrong-movement rate means the model
# emitted zero movement commands, so the denominator is zero—not a measured 0% error rate.
paired_gate_rows = []
for name, content in paired_results.items():
    pred, conf = content["prediction"], content["confidence"]
    for threshold in (0.60, 0.70):
        accepted = conf >= threshold
        movement = accepted & np.isin(pred, [1, 2])
        true_rest = y_val == 0
        wrong_movement = movement & (pred != y_val)
        paired_gate_rows.append({
            "model": name,
            "threshold": threshold,
            "accepted_output_rate": float(accepted.mean()),
            "no_command_rate": float((~accepted).mean()),
            "movement_commands": int(movement.sum()),
            "false_movement_on_true_rest_rate": float((movement & true_rest).sum() / true_rest.sum()) if true_rest.any() else np.nan,
            "wrong_movement_command_rate": float(wrong_movement.sum() / movement.sum()) if movement.sum() else np.nan,
        })
paired_gate_comparison = pd.DataFrame(paired_gate_rows)
print("Confidence-gate results for the paired models:")
display(paired_gate_comparison.round(4))
paired_gate_comparison.to_csv(OUTPUT_DIR / "class_weighted_gate_comparison.csv", index=False)

# Save both comparison candidates separately; keep the original first-run handoff unchanged.
ARTIFACT_DIR.mkdir(exist_ok=True)
for name, content in paired_results.items():
    slug = "unweighted" if "unweighted" in name else "class_weighted"
    weights_file = ARTIFACT_DIR / f"neuromotion_eegnet_{slug}_comparison_state_dict.pt"
    state_dict_cpu = {key: tensor.detach().cpu() for key, tensor in content["model"].module_.state_dict().items()}
    torch.save(state_dict_cpu, weights_file)
    print("Saved comparison weights:", weights_file.resolve())

weighted_config = {
    "experiment": "paired unweighted versus class-weighted cross-entropy",
    "single_intended_change": "CrossEntropyLoss class weights inversely proportional to training class counts",
    "class_names": CLASS_NAMES,
    "class_weights_by_class": {CLASS_NAMES[i]: float(class_weight_values[i]) for i in range(len(CLASS_NAMES))},
    "training_run": list(TRAIN_RUNS),
    "validation_run": list(VALIDATION_RUNS),
    "reserved_test_run_not_loaded": list(RESERVED_TEST_RUNS),
    "seed": SEED,
    "configuration": INITIAL_CONFIG,
    "note": "Development comparison only. Do not use R12 until the model and gate are frozen.",
}
with open(ARTIFACT_DIR / "neuromotion_class_weighted_comparison_config.json", "w", encoding="utf-8") as f:
    json.dump(weighted_config, f, indent=2)

# Save paired per-window predictions for auditability and error analysis.
paired_prediction_frame = meta_val.copy().reset_index(drop=True)
paired_prediction_frame["true_class"] = [CLASS_NAMES[int(v)] for v in y_val]
for name, content in paired_results.items():
    slug = "unweighted" if "unweighted" in name else "class_weighted"
    paired_prediction_frame[f"{slug}_prediction"] = [CLASS_NAMES[int(v)] for v in content["prediction"]]
    paired_prediction_frame[f"{slug}_confidence"] = content["confidence"]
    paired_prediction_frame[f"{slug}_correct"] = content["prediction"] == y_val
paired_prediction_frame.to_csv(OUTPUT_DIR / "class_weighted_validation_predictions.csv", index=False)

print("Saved comparison metrics and predictions to:", OUTPUT_DIR.resolve())
print("Interpretation: class weighting only helps if the measured validation class metrics/predictions improve without an unacceptable command-safety trade-off. Do not claim an improvement unless the printed tables support it.")


Training counts by class: {'Rest': 15, 'Left Hand': 8, 'Right Hand': 7}
Class weights: {'Rest': 0.6667, 'Left Hand': 1.25, 'Right Hand': 1.4286}
Paired comparison on the same R08 validation windows (not final-test results):


,model,validation_windows,accuracy,balanced_accuracy,macro_f1,max_confidence,accepted_rate_at_0.60,accepted_rate_at_0.70,training_seconds,predicted_Rest,predicted_Left_Hand,predicted_Right_Hand
0,EEGNet unweighted (paired baseline),30,0.5000,0.3333,0.2222,0.3762,0.0,0.0,0.2256,30,0,0
1,EEGNet class-weighted CE,30,0.4667,0.3948,0.3909,0.3535,0.0,0.0,0.1595,18,7,5



Confusion matrix — EEGNet unweighted (paired baseline)


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,15,0,0
true_Left Hand,8,0,0
true_Right Hand,7,0,0


              precision    recall  f1-score   support

        Rest       0.50      1.00      0.67        15
   Left Hand       0.00      0.00      0.00         8
  Right Hand       0.00      0.00      0.00         7

    accuracy                           0.50        30
   macro avg       0.17      0.33      0.22        30
weighted avg       0.25      0.50      0.33        30


Confusion matrix — EEGNet class-weighted CE


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,10,2,3
true_Left Hand,4,3,1
true_Right Hand,4,2,1


              precision    recall  f1-score   support

        Rest       0.56      0.67      0.61        15
   Left Hand       0.43      0.38      0.40         8
  Right Hand       0.20      0.14      0.17         7

    accuracy                           0.47        30
   macro avg       0.39      0.39      0.39        30
weighted avg       0.44      0.47      0.45        30

Confidence-gate results for the paired models:


,model,threshold,accepted_output_rate,no_command_rate,movement_commands,false_movement_on_true_rest_rate,wrong_movement_command_rate
0,EEGNet unweighted (paired baseline),0.6,0.0,1.0,0,0.0,NaN
1,EEGNet unweighted (paired baseline),0.7,0.0,1.0,0,0.0,NaN
2,EEGNet class-weighted CE,0.6,0.0,1.0,0,0.0,NaN
3,EEGNet class-weighted CE,0.7,0.0,1.0,0,0.0,NaN


Saved comparison weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_unweighted_comparison_state_dict.pt
Saved comparison weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_class_weighted_comparison_state_dict.pt
Saved comparison metrics and predictions to: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs
Interpretation: class weighting only helps if the measured validation class metrics/predictions improve without an unacceptable command-safety trade-off. Do not claim an improvement unless the printed tables support it.


## Limitations and next actions

1. The initial measured run is a failed baseline: it predicted Rest for every R08 validation window (accuracy 0.50 equals the majority baseline; balanced accuracy 0.333; macro-F1 0.222). 
2. This is a small, within-subject development experiment: only 30 windows are in each run. The resulting metric variance will be high and should not be generalized to other people.
3. `R12` is reserved for the future final test and must remain unused until the model/configuration and confidence threshold are frozen.
4. The project must compare EEGNet against a simpler CSP + LDA baseline and/or ShallowFBCSPNet on the same development split before deciding that the neural approach is preferable.
5. Confidence is not calibrated probability of correctness. The confidence gate is a heuristic until calibrated on appropriate validation data. High-confidence wrong predictions remain possible; the initial 0.60/0.70 gate comparison is uninformative because every confidence was below 0.60.
6. The corrected latency cell must be rerun because it now warms up the model and synchronizes CUDA around timed calls. The measurement still excludes EDF reading, streaming filters, UI rendering, and replay orchestration.
7. Do not interpret model predictions as real prosthetic control. The current deliverable supports offline recorded-EEG replay only.

## References

- Braindecode documentation, *Simple training on MNE epochs*: https://braindecode.org/stable/auto_examples/model_building/plot_basic_training_epochs.html
- Braindecode API, `EEGNet`: https://braindecode.org/stable/generated/braindecode.models.EEGNet.html
- Braindecode API, `EEGClassifier`: https://braindecode.org/stable/generated/braindecode.classifier.EEGClassifier.html
- PhysioNet EEG Motor Movement/Imagery Dataset v1.0.0: https://physionet.org/content/eegmmidb/1.0.0/ (DOI: 10.13026/C28G6P)
- Schalk, G. (2009). *EEG Motor Movement/Imagery Dataset (version 1.0.0).* PhysioNet.